<a href="https://colab.research.google.com/github/GEO-HACK/watchtower-ml/blob/main/notebooks/random_trainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import os
os.listdir('/content/drive/MyDrive/CICIDS')

['X_train.npy',
 'X_test.npy',
 'y_train.npy',
 'y_test.npy',
 'preprocessing_pipeline.pkl',
 'label_encoder.pkl']

In [13]:
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay,
    f1_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("imports done")

imports done


loading the data

In [14]:
BASE = '/content/drive/MyDrive/CICIDS/'

X_train = np.load(BASE + 'X_train.npy')
X_test  = np.load(BASE + 'X_test.npy')
y_train = np.load(BASE + 'y_train.npy')
y_test  = np.load(BASE + 'y_test.npy')

In [15]:
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")

X_train shape : (1725584, 68)
X_test shape  : (431397, 68)


Model definition

In [20]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth = None,
    min_samples_split=10,
    min_samples_leaf=5,

    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

In [17]:
print("Running stratified k-fold cv( k=5) ...")

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    rf,X_train,y_train,cv=cv,scoring='f1_macro',
    n_jobs=-1
)

print(f"\nCV Macro F1 per fold : {cv_scores.round(4)}")
print(f"Mean  : {cv_scores.mean():.4f}")
print(f"Std   : {cv_scores.std():.4f}")

Running stratified k-fold cv( k=5) ...


KeyboardInterrupt: 

In [21]:
print("Training Random Forest on full training set...")

rf.fit(X_train, y_train)

print("✅ Training complete")

Training Random Forest on full training set...
✅ Training complete


perform evaluation opn test data

In [22]:
y_pred = rf.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"Overall Macro F1: {macro_f1:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    346659
           1       0.70      0.97      0.81       390
           2       1.00      1.00      1.00     25603
           3       0.99      1.00      0.99      2057
           4       1.00      1.00      1.00     34570
           5       1.00      0.99      0.99      1046
           6       0.99      0.99      0.99      1077
           7       1.00      1.00      1.00      1187
           8       1.00      1.00      1.00     18164
           9       1.00      1.00      1.00       644

    accuracy                           1.00    431397
   macro avg       0.97      0.99      0.98    431397
weighted avg       1.00      1.00      1.00    431397

Overall Macro F1: 0.9791
